# 1. Raw data 가져오기

In [8]:
import pandas as pd

df = pd.read_excel("raw_data/20250219_시사경제용어사전.xlsx")
df.head()

/opt/anaconda3/envs/krx/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,순번,주제,용어,설명
0,1,사회,0.5인 가구,싱글족 가운데 두 곳 이상에 거처를 두거나 잦은 여행과 출장 등으로 오랫동안 집을 ...
1,2,경영,1인 창조기업,"개인이 사장이면서 직원인 기업을 의미한다. 자신이 가진 '지식, 경험, 기술' 등을..."
2,3,경제,1인당 국민소득,국민소득을 총국민 수로 나눈 값. 해당 국가의 소득 수준을 보여주는 가장 대표적인 ...
3,4,과학,20-20-20 계획,"유럽연합(EU)이 2020년까지 온실가스 20% 감축, 에너지효율 20% 개선, 신..."
4,5,금융,2차 시장(Secondary Market),"2차 시장은 처음 발행된 증권, 채권 등이 거래되는 발행시장과 구분되며, 이미 발행..."


In [9]:
finance_df = df[df["주제"] == "금융"]

print(len(finance_df))
finance_df.head()

816


,순번,주제,용어,설명
4,5,금융,2차 시장(Secondary Market),"2차 시장은 처음 발행된 증권, 채권 등이 거래되는 발행시장과 구분되며, 이미 발행..."
12,13,금융,5일선,"주가의 평균치를 이어놓은 이동평균선에서 사용되는 말로, 5일선이란 5일동안의 평균주..."
16,17,금융,ABCP(Asset Backed Commercial Paper),Asset Backed Commercial Paper의 약어. 유동화전문회사(SPC...
19,20,금융,AMA(Auto Management Account),고객이 설정한 조건에 따라 상대적으로 고금리를 주는 예금이나 증권사로 자동이체·관리...
24,25,금융,"At The Money(ATM, 앳 더 머니)",‘앳 더 머니’ 상황은 옵션의 행사가격이 기초자산의 시장 가격과 동일할 때를 가리킨...


# 2. 합성 데이터 생성

In [5]:
import os
import json
import numpy as np
from openai import OpenAI
import traceback
from dotenv import load_dotenv
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd


# .env 파일 로드
load_dotenv('.env')

# API_KEY 값을 가져옴
openai_api_key = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = openai_api_key

# Upstage API 클라이언트 설정

client_gpt = OpenAI()

In [6]:
import json

def generate_qa_from_context(context, keyword):
    """
    Generates 5 high-quality question-answer pairs from a keyword and its explaining context.
    """
    # Define context and keyword message
    context_text = f"[KEYWORD]\n{keyword}\n[CONTEXT]\n{context}"
    
    # Define prompt to ensure clear QA pair generation around the keyword and context
    prompt = f"""
    You are a skilled question generator. Using the keyword and its explaining context provided, create exactly 5 high-quality question-answer pairs in Korean. Each question should focus on expanding the reader's understanding of the keyword by using the context. Follow these structures to ensure the output is clear, informative, and detailed.

    **Keyword and Context**:
    {context_text}

    **Question Structures**:
    - **Summarization**:
        - question: Create a question that summarizes the main idea of the context related to the keyword.
        - Example Format: "이 문장의 요약은 무엇인가요?"
        - answer: Summarize the core idea of how the context explains the keyword.

    - **Topic Analysis**:
        - question: Formulate a question identifying the main topic or use of the keyword within the context.
        - Example Format: "{keyword}의 주요 사용 또는 의미는 무엇인가요?"
        - answer: Provide an answer that clarifies the main topic or role of the keyword based on the context.

    - **Detailed Explanation**:
        - question: Ask for a more detailed explanation of a specific point mentioned in the context about the keyword.
        - Example Format: "{keyword}가 사용되는 방식에 대해 자세히 설명해 주세요."
        - answer: Elaborate on a specific detail within the context that relates to the keyword.

    - **Practical Usage**:
        - question: Pose a question about how the keyword might be used in practical situations, as described in the context.
        - Example Format: "{keyword}는 실제로 어떻게 사용될 수 있나요?"
        - answer: Describe a practical usage or example from the context.

    - **Commonsense Reasoning**:
        - question: Create a question asking about the implications or importance of the keyword in a larger context.
        - Example Format: "{keyword}의 중요성은 무엇인가요?"
        - answer: Explain why the keyword is significant, based on information from the context.

    Each question and answer should follow these templates and be provided in JSON format, in Korean.
    """
    
    # Send prompt to the model
    response = client_gpt.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{'role': 'user', 'content': prompt}],
        response_format={"type": "json_object"}
    )
    
    # Process the response
    qa_output = response.choices[0].message.content
    
    return qa_output


In [41]:
sample_data = finance_df.iloc[1]
sample_response = generate_qa_from_context(sample_data["설명"], sample_data['용어'])
print(sample_response)

{
  "questions": [
    {
      "question": "이 문장의 요약은 무엇인가요?",
      "answer": "5일선은 주가의 평균치를 이어놓은 이동평균선 중 하나로, 최근 5일 동안의 평균 주가를 시각적으로 나타내어 주가 흐름을 파악하고 예측하는 데 도움을 준다."
    },
    {
      "question": "5일선의 주요 사용 또는 의미는 무엇인가요?",
      "answer": "5일선은 단기 시세 흐름을 분석하고 예측하기 위해 사용되며, 다른 이동평균선인 10일선이나 20일선과 함께 주가의 변동성을 평가하는 역할을 한다."
    },
    {
      "question": "5일선이 사용되는 방식에 대해 자세히 설명해 주세요.",
      "answer": "5일선은 매일의 주가를 바탕으로 최근 5일간의 평균을 구하여 그 값을 연결한 선으로, 투자자들이 주가의 단기적인 추세를 쉽게 분석할 수 있도록 도와준다."
    },
    {
      "question": "5일선은 실제로 어떻게 사용될 수 있나요?",
      "answer": "투자자들은 5일선을 차트에 표시하여 주가의 상승이나 하락 추세를 판단하고, 매매 결정을 내릴 때 참고하는 지표로 활용한다."
    },
    {
      "question": "5일선의 중요성은 무엇인가요?",
      "answer": "5일선은 주가의 단기적인 흐름을 분석하는 데 유용하며, 투자자들에게 빠른 매매 기회를 제공하고 주식 거래의 의사결정을 보다 효율적으로 만들어준다."
    }
  ]
}


# 3. formatted 데이터 만들기

```
### Instruction: This task focuses on understanding key finance terms. For each term, read the context carefully, then answer questions about its definition, typical usage, practical examples, and relevance in finance. Each answer will deepen your understanding of the term’s role in the financial field.

### Term: {keyword}

### Context:  {context}
### Questions:
    1. {question1}
        - {answer1}
    2. {question2}
        - {answer2}
    3. {question3}
        - {answer3}
    4. {question4}
        - {answer4}
    5. {question5}
        - {answer5}
```

In [58]:
import json

def item_to_text(term, context, response):
    response = json.loads(response)

    # questions 라고 key 명을 안하는 경우 처리하는 코드
    if len(response.keys()) == 1:
        response = {"questions": list(response.values())[0]}
    else:
        print(response)
        return None

    try :
        questions = response["questions"]
        question_format_text = '\n'.join([f"\t{idx+1}. {item['question']}\n\t\t- {item['answer']}" for idx, item in enumerate(questions)])

        formmatted_text = f"""### Instruction: This task focuses on understanding key finance terms. For each term, read the context carefully, then answer questions about its definition, typical usage, practical examples, and relevance in finance. Each answer will deepen your understanding of the term’s role in the financial field.

### Term: {term}

### Context: {context}

### Questions: 
{question_format_text}
    """
        return formmatted_text

    except:
        print(response)
        return None

In [60]:
from tqdm import tqdm
import pandas as pd
import time

responses = []

for idx, row in tqdm(finance_df.iterrows(), total=finance_df.shape[0]):
    # 파일에 저장
    if idx % 5 == 0 and len(responses) != 0:
        temp_df = pd.DataFrame(responses)
        temp_df.to_csv("datasets/formatted_finance_2.csv", mode='a', header=False, index=False, quoting=csv.QUOTE_ALL)
        responses = []

    response = generate_qa_from_context(row['설명'], row['용어'])
    time.sleep(1)
    
    item_formmated_text = item_to_text(row['용어'], row['설명'], response)
    
    # model이 답변 형식을 틀린 경우
    if not item_formmated_text:
        continue

    responses.append({"formmated_text":item_formmated_text})

temp_df = pd.DataFrame(responses)
temp_df.to_csv("datasets/formatted_finance_2.csv", mode='a', header=False, index=False)

 17%|█▋        | 141/816 [20:35<1:39:55,  8.88s/it]

{'questions': [{'질문': '이 문장의 요약은 무엇인가요?', '답변': '구주매출은 기존 주주가 보유한 주식의 일부를 일반인에게 공개적으로 판매하는 과정을 말하며, 이는 투자금 회수와 주식 투자 및 경영권 인수와 관련이 있다.'}, {'질문': '구주매출의 주요 사용 또는 의미는 무엇인가요?', '답변': '구주매출은 기존 주주가 보유한 주식의 일부를 공개적으로 매각하여 투자금을 회수하는 방식으로 사용되며, 새로운 투자자에게는 주식 투자 및 경영권 참여의 기회를 제공한다.'}, {'질문': '구주매출가 사용되는 방식에 대해 자세히 설명해 주세요.', '답변': '구주매출은 대주주나 일반 주주가 보유하고 있는 주식의 일부를 공식적으로 시장에 내놓고 판매하는 과정이며, 이 과정에서 양도인은 필요 자금을 회수하고 양수인은 주식 투자 기회를 가지게 된다.'}, {'질문': '구주매출는 실제로 어떻게 사용될 수 있나요?', '답변': '구주매출은 기업이 성장하거나 자금 조달이 필요할 때 대주주가 자신이 가진 주식의 일부를 시장에 판매하여 자금을 확보하는 데 사용될 수 있다.'}, {'질문': '구주매출의 중요성은 무엇인가요?', '답변': '구주매출은 기존 주주의 투자금을 회수하면서 동시에 새로운 투자자에게 주식 투자 기회를 제공하므로, 기업의 자본 구조와 주식 시장 유동성 유지에 중요한 역할을 한다.'}]}


 35%|███▍      | 284/816 [41:19<1:17:25,  8.73s/it]


KeyboardInterrupt: 

# 4. 생성된 파일 확인

In [2]:
import pandas as pd

finance_df = pd.read_csv('datasets/formatted_finance.csv')
finance_df.head()

,formatted_text
0,### Instruction: This task focuses on understa...
1,### Instruction: This task focuses on understa...
2,### Instruction: This task focuses on understa...
3,### Instruction: This task focuses on understa...
4,### Instruction: This task focuses on understa...
